In [1]:
# Cell 1: Clone repository và tải các thư mục cần thiết
import os

repo_url = "https://github.com/nta2112/OW_OVD-An-custom.git"
working_dir = "/kaggle/working/OW_OVD"

if not os.path.exists(working_dir):
    print("Cloning repository...")
    !git clone {repo_url} {working_dir}
else:
    print("Repository đã tồn tại. Đang cập nhật (pull)...")
    %cd {working_dir}
    !git pull

%cd {working_dir}

# Clone thư mục con mmyolo vào third_party nếu chưa có
if not os.path.exists("third_party/mmyolo"):
    print("Cloning mmyolo...")
    !git clone https://github.com/open-mmlab/mmyolo.git third_party/mmyolo
else:
    print("mmyolo đã tồn tại.")


Cloning repository...
Cloning into '/kaggle/working/OW_OVD'...
remote: Enumerating objects: 715, done.
remote: Counting objects: 100% (88/88), done.
remote: Compressing objects: 100% (58/58), done.
remote: Total 715 (delta 58), reused 57 (delta 30), pack-reused 627 (from 1)
Receiving objects: 100% (715/715), 1.34 MiB | 8.50 MiB/s, done.
Resolving deltas: 100% (453/453), done.
/kaggle/working/OW_OVD
Cloning mmyolo...
Cloning into 'third_party/mmyolo'...
remote: Enumerating objects: 4968, done.
remote: Counting objects: 100% (1341/1341), done.
remote: Compressing objects: 100% (294/294), done.
remote: Total 4968 (delta 1133), reused 1047 (delta 1047), pack-reused 3627 (from 1)
Receiving objects: 100% (4968/4968), 3.62 MiB | 14.65 MiB/s, done.
Resolving deltas: 100% (3216/3216), done.


In [ ]:
# Cell 2: Cài đặt toàn bộ môi trường và các thư viện trong một bước
# LƯU Ý: Không import torch trước khi chạy cell này để tránh xung đột bộ nhớ RAM!
import sys

# 1. Hạ cấp PyTorch & Torchvision xuống bản ổn định 2.4.0
!python -m pip install torch==2.4.0+cu121 torchvision==0.19.0+cu121 --extra-index-url https://download.pytorch.org/whl/cu121

# 2. Pin Pillow về phiên bản tương thích với torchvision trước khi import transformers
!python -m pip install --no-cache-dir --force-reinstall "pillow==10.4.0"

# 3. Clear stale module cache inside the current kernel before re-importing Pillow/transformers
for name in ['PIL', 'PIL.Image', 'PIL._typing', 'torchvision', 'transformers']:
    sys.modules.pop(name, None)

import PIL
print(f"Pillow version after pinning: {PIL.__version__}")

# 4. Cài đặt MMCV theo fallback an toàn cho Python 3.12 / Kaggle
!python -m pip install --no-cache-dir --force-reinstall "mmcv-lite==2.0.1"

# 5. Cài đặt các thư viện phụ trợ
!python -m pip install matplotlib pycocotools terminaltables mmengine prettytable wcwidth open_clip_torch transformers

print(f"mmcv version: {mmcv.__version__}")

# 6. Cài đặt MMDetection và MMYOLOimport mmcv

!python -m pip install "mmdet>=3.1.0" --no-deps# 7. Smoke test chứng minh mmcv đã import được

!python -m pip install --no-build-isolation --no-deps third_party/mmyolo

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 799.0/799.0 MB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 94.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 73.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 41.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 100.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 11.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 34.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 15.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━

In [3]:
# # Cell 3: Cài đặt các thư viện bổ trợ và cài đặt mmdet/mmyolo
# !pip install matplotlib pycocotools terminaltables mmengine prettytable wcwidth open_clip_torch transformers

# # Cài đặt mmdet và mmyolo bỏ qua kiểm tra phiên bản setuptools
# !pip install "mmdet>=3.1.0" --no-deps
# !pip install --no-build-isolation --no-deps third_party/mmyolo


In [4]:
# Cell 4: Vá lỗi giới hạn phiên bản MMCV trong mmdet và mmyolo
# import importlib.util

# # Vá lỗi MMDetection
# spec = importlib.util.find_spec('mmdet')
# if spec is not None and spec.origin is not None:
#     init_file = spec.origin
#     with open(init_file, 'r') as f:
#         content = f.read()
#     if "mmcv_maximum_version = '2.2.0'" in content:
#         content = content.replace("mmcv_maximum_version = '2.2.0'", "mmcv_maximum_version = '2.3.0'")
#         with open(init_file, 'w') as f:
#             f.write(content)
#         print("Đã vá lỗi check MMCV của MMDetection thành công!")
#     else:
#         print("Không cần vá hoặc MMDetection đã được vá trước đó.")

# # Vá lỗi MMYOLO
# spec = importlib.util.find_spec('mmyolo')
# if spec is not None and spec.origin is not None:
#     init_file = spec.origin
#     with open(init_file, 'r') as f:
#         content = f.read()
#     if "mmcv_maximum_version = '2.1.0'" in content:
#         content = content.replace("mmcv_maximum_version = '2.1.0'", "mmcv_maximum_version = '2.3.0'")
#         with open(init_file, 'w') as f:
#             f.write(content)
#         print("Đã vá lỗi check MMCV của MMYOLO thành công!")
#     else:
#         print("Không cần vá hoặc MMYOLO đã được vá trước đó.")

# ----------------------------------------------------
# code phiên bản mmolo cho base gt (ablation studies)
# import importlib.util
# import re

# for pkg in ['mmdet', 'mmyolo']:
#     spec = importlib.util.find_spec(pkg)
#     if spec is not None and spec.origin is not None:
#         init_file = spec.origin
#         with open(init_file, 'r') as f:
#             content = f.read()
        
#         # Tự động thay thế mọi mmcv_maximum_version = '...' thành '2.3.0'
#         new_content = re.sub(r"mmcv_maximum_version\s*=\s*['\"].*?['\"]", "mmcv_maximum_version = '2.3.0'", content)
        
#         if new_content != content:
#             with open(init_file, 'w') as f:
#                 f.write(new_content)
#             print(f"Đã vá lỗi check MMCV của {pkg} thành công!")
#         else:
#             print(f"Không cần vá hoặc {pkg} đã được vá trước đó.")
# Cell 3: Vá lỗi giới hạn phiên bản MMCV cho cả thư viện hệ thống và thư mục third_party cục bộ
import os
import re
import importlib.util

# Danh sách các đường dẫn cần quét và vá
files_to_patch = [
    # Thư mục cục bộ trong third_party (Nơi train.py ưu tiên load trước)
    'third_party/mmyolo/mmyolo/__init__.py',
]

# Tự động tìm thêm đường dẫn cài đặt hệ thống của mmdet và mmyolo
for pkg in ['mmdet', 'mmyolo']:
    spec = importlib.util.find_spec(pkg)
    if spec is not None and spec.origin is not None:
        files_to_patch.append(spec.origin)

# Thực hiện vá lỗi
for filepath in files_to_patch:
    if os.path.exists(filepath):
        with open(filepath, 'r', encoding='utf-8') as f:
            content = f.read()
        
        # Thay thế giới hạn mmcv_maximum_version thành '2.3.0'
        new_content = re.sub(
            r"mmcv_maximum_version\s*=\s*['\"].*?['\"]", 
            "mmcv_maximum_version = '2.3.0'", 
            content
        )
        
        if new_content != content:
            with open(filepath, 'w', encoding='utf-8') as f:
                f.write(new_content)
            print(f"-> Đã vá lỗi check MMCV thành công cho: {filepath}")
        else:
            print(f"-> {filepath} đã được vá hoặc không cần sửa.")
    else:
        print(f"-> Đường dẫn không tồn tại (bỏ qua): {filepath}")


-> Đã vá lỗi check MMCV thành công cho: third_party/mmyolo/mmyolo/__init__.py
-> Đã vá lỗi check MMCV thành công cho: /usr/local/lib/python3.12/dist-packages/mmdet/__init__.py
-> Đã vá lỗi check MMCV thành công cho: /usr/local/lib/python3.12/dist-packages/mmyolo/__init__.py


In [5]:
# Monkeypatch để bỏ qua kiểm tra phiên bản PyTorch >= 2.6 khi nạp file .bin
import transformers.utils.import_utils
transformers.utils.import_utils.check_torch_load_is_safe = lambda: None

In [6]:
# # Cell 5: Tạo thư mục, tải pretrain weights và sinh các file embeddings + XML annotations cần thiết
# import json
# import torch
# import numpy as np
# import os
# from transformers import AutoTokenizer, CLIPTextModelWithProjection

# # 1. Tạo các thư mục lưu dữ liệu
# os.makedirs('pretrained_models', exist_ok=True)
# os.makedirs('data/IP102', exist_ok=True)
# os.makedirs('data/texts/IP102', exist_ok=True)

# # 1.5. Copy dataset từ read-only network mount sang fast local SSD /tmp
# print("-> Đang copy dataset từ /kaggle/input sang /tmp để tối ưu tốc độ đọc (I/O)...")
# os.makedirs('/tmp/ip02-dataset/classification', exist_ok=True)
# if len(os.listdir('/tmp/ip02-dataset/classification')) == 0:
#     print("-> Thư mục /tmp trống. Bắt đầu copy qua tar pipeline...")
#     !tar -cf - -C /kaggle/input/datasets/rtlmhjbn/ip02-dataset/classification . | tar -xf - -C /tmp/ip02-dataset/classification
#     if len(os.listdir('/tmp/ip02-dataset/classification')) == 0:
#         print("-> Tar pipeline thất bại. Thử lại bằng cp -r...")
#         !cp -r /kaggle/input/datasets/rtlmhjbn/ip02-dataset/classification/* /tmp/ip02-dataset/classification/
#     print(f"-> Copy hoàn tất! Số ảnh trong /tmp: {len(os.listdir('/tmp/ip02-dataset/classification'))}")
# else:
#     print(f"-> Dataset đã tồn tại trong /tmp với {len(os.listdir('/tmp/ip02-dataset/classification'))} files. Không cần copy lại.")

# # 2. Tải pretrain weights của YOLO-World
# print("-> Đang tải weights...")
# weights_path = 'pretrained_models/yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth'
# if not os.path.exists(weights_path):
#     !wget -O {weights_path} \
#       https://huggingface.co/wondervictor/YOLO-World/resolve/main/yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth
# else:
#     print("Pretrained weights đã tồn tại.")

# # 3. Đọc tên các class từ file annotations
# ann_path = '/kaggle/input/datasets/eljazouly/ip102-coco-annotations/coco_annotations/train.json'
# with open(ann_path, 'r') as f:
#     coco_data = json.load(f)

# categories = sorted(coco_data['categories'], key=lambda x: x['id'])
# class_names = [cat['name'] for cat in categories]
# num_classes = len(class_names)
# print(f"Tổng số lớp học (classes): {num_classes}")

# # 4. Lưu danh sách class vào file class_texts.json
# class_texts = [[name] for name in class_names]
# with open('data/texts/IP102/class_texts.json', 'w') as f:
#     json.dump(class_texts, f)

# # 5. Trích xuất text embeddings bằng CLIP
# print("-> Đang sinh class embeddings từ CLIP model...")
# model_name = 'openai/clip-vit-base-patch32'
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = CLIPTextModelWithProjection.from_pretrained(model_name, use_safetensors=True)
# model.eval()

# embeddings = []
# with torch.no_grad():
#     for name in class_names:
#         inputs = tokenizer(name, padding=True, return_tensors="pt")
#         outputs = model(**inputs)
#         embed = outputs.text_embeds[0].cpu().numpy()
#         embed = embed / np.linalg.norm(embed)
#         embeddings.append(embed)

# np.save('data/IP102/ip102_gt_embeddings.npy', np.array(embeddings))

# # 6. Sinh file task_att_1_embeddings.pth chứa cả att_embedding và att_text
# print("-> Đang sinh file dummy attribute embeddings...")
# num_att = num_classes * 25
# torch.save({
#     'att_embedding': torch.zeros(num_att, 512),
#     'att_text': [f"att_{i}" for i in range(num_att)]
# }, 'data/IP102/task_att_1_embeddings.pth')

# # 7. Sinh file phân phối mẫu dummy
# thrs = [0.55]
# pos_dist = [{att_i: torch.zeros(10000) for att_i in range(num_att)} for _ in thrs]
# neg_dist = [{att_i: torch.zeros(10000) for att_i in range(num_att)} for _ in thrs]
# torch.save({
#     'positive_distributions': pos_dist,
#     'negative_distributions': neg_dist
# }, 'data/IP102/mowod_distribution_sim1.pth')

# # 8. Chuyển đổi COCO annotations sang PASCAL VOC (XML) siêu tốc phục vụ OWODEvaluator
# print("-> Đang chuyển đổi COCO JSON sang cấu trúc PASCAL VOC...")
# voc_root = 'data/IP102/voc/'
# anno_dir = os.path.join(voc_root, "Annotations")
# imageset_dir = os.path.join(voc_root, "ImageSets", "Main", "mowod")
# os.makedirs(anno_dir, exist_ok=True)
# os.makedirs(imageset_dir, exist_ok=True)

# # Gom các annotation theo ID ảnh
# image_annos = {}
# for ann in coco_data['annotations']:
#     image_annos.setdefault(ann['image_id'], []).append(ann)

# # Tạo ánh xạ nhãn an toàn tự động phát hiện lệch chỉ số
# cat_map = {cat['id']: cat['name'] for cat in categories}
# anno_ids = {ann['category_id'] for ann in coco_data['annotations']}

# if 0 in anno_ids and 0 not in cat_map:
#     print("-> Phát hiện nhãn đánh số 0-indexed trong annotations. Đang tự động khớp chỉ số...")
#     # Nếu nhãn đánh số từ 0 đến N-1
#     if max(anno_ids) == len(class_names) - 1:
#         cat_map = {i: class_names[i] for i in range(len(class_names))}
#     else:
#         # Dự phòng khẩn cấp
#         cat_map[0] = class_names[0] if len(class_names) > 0 else 'unknown'
#         for i in range(1, len(class_names) + 1):
#             if i not in cat_map:
#                 cat_map[i] = class_names[i-1] if i-1 < len(class_names) else 'unknown'

# image_names = []

# for img in coco_data['images']:
#     img_id = img['id']
#     file_name = img['file_name']
#     width = img.get('width', 640)
#     height = img.get('height', 640)
    
#     img_stem = os.path.splitext(os.path.basename(file_name))[0]
#     image_names.append(img_stem)
    
#     # Tạo chuỗi XML tối ưu
#     xml_content = f"""<annotation>
#   <filename>{os.path.basename(file_name)}</filename>
#   <size>
#     <width>{width}</width>
#     <height>{height}</height>
#     <depth>3</depth>
#   </size>
# """
#     annos = image_annos.get(img_id, [])
#     for ann in annos:
#         bbox = ann['bbox'] # [x, y, w, h]
#         cat_name = cat_map.get(ann['category_id'], 'unknown')
        
#         # Chuyển đổi [x, y, w, h] sang [xmin, ymin, xmax, ymax]
#         xmin = max(1, int(round(bbox[0])))
#         ymin = max(1, int(round(bbox[1])))
#         xmax = max(xmin + 1, int(round(bbox[0] + bbox[2])))
#         ymax = max(ymin + 1, int(round(bbox[1] + bbox[3])))
        
#         xml_content += f"""  <object>
#     <name>{cat_name}</name>
#     <pose>Unspecified</pose>
#     <truncated>0</truncated>
#     <difficult>0</difficult>
#     <bndbox>
#       <xmin>{xmin}</xmin>
#       <ymin>{ymin}</ymin>
#       <xmax>{xmax}</xmax>
#       <ymax>{ymax}</ymax>
#     </bndbox>
#   </object>
# """
#     xml_content += "</annotation>"
    
#     xml_path = os.path.join(anno_dir, f"{img_stem}.xml")
#     with open(xml_path, 'w', encoding='utf-8') as f_xml:
#         f_xml.write(xml_content)

# # Ghi danh sách ảnh vào file
# txt_path = os.path.join(imageset_dir, "all_task_test.txt")
# with open(txt_path, 'w') as f_txt:
#     f_txt.write("\n".join(image_names))

# print(f"Thành công chuyển đổi {len(image_names)} nhãn sang VOC XML tại {voc_root}!")


In [ ]:
# Cell 5: Tạo thư mục, nạp pretrain weights từ Kaggle Input và sinh các file embeddings + XML annotations cần thiết
import os
import glob
import json
import sys
import importlib
import torch
import numpy as np

# 0) Force-reinstall Pillow to a stable version and clear stale imports from the kernel
!python -m pip install --no-cache-dir --force-reinstall "pillow==10.4.0"
for name in [
    'PIL', 'PIL.Image', 'PIL.ImageFont', 'PIL.ImageDraw', 'PIL.ImageText',
    'PIL._typing', 'torchvision', 'torchvision.transforms', 'transformers',
    'transformers.utils.import_utils', 'transformers.modeling_utils'
]:
    sys.modules.pop(name, None)

import PIL
from PIL import Image
import PIL._typing
print(f"Pillow version after pinning: {PIL.__version__}")

# 1) Safe import of CLIP tokenizer/model after Pillow relink
import transformers.utils.import_utils
from transformers import AutoTokenizer, CLIPTextModelWithProjection

# Monkeypatch để bỏ qua yêu cầu kiểm tra phiên bản PyTorch 2.6 (Khắc phục lỗi CVE-2025-32434)
transformers.utils.import_utils.check_torch_load_is_safe = lambda: None

# 2) Tạo các thư mục lưu dữ liệu
os.makedirs('pretrained_models', exist_ok=True)
os.makedirs('pretrained_models/clip-vit-base-patch32', exist_ok=True)
os.makedirs('data/IP102', exist_ok=True)
os.makedirs('data/texts/IP102', exist_ok=True)

# 3) Tạo Symlink thay vì copy Dataset ảnh
print("-> Đang tạo liên kết mềm (Symlink) cho Dataset...")
!rm -rf /tmp/ip02-dataset
os.makedirs('/tmp/ip02-dataset', exist_ok=True)
!ln -sf /kaggle/input/datasets/rtlmhjbn/ip02-dataset/classification /tmp/ip02-dataset/classification
print("-> Tạo Symlink thành công!")

# 4) Nạp pretrain weights của YOLO-World từ Kaggle Input
print("-> Đang nạp weights YOLO-World...")
weights_path = 'pretrained_models/yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth'
if not os.path.exists(weights_path):
    kaggle_input_weights = glob.glob('/kaggle/input/**/yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth', recursive=True)
    if kaggle_input_weights:
        !cp {kaggle_input_weights[0]} pretrained_models/
        print("-> Đã nạp weights YOLO-World từ Kaggle Input thành công!")
    else:
        raise FileNotFoundError("Không tìm thấy weights YOLO-World trong Kaggle Input! Vui lòng thêm Dataset chứa file weights.")
else:
    print("Pretrained weights đã tồn tại.")

# 5) Đọc tên các class từ file annotations
ann_path = '/kaggle/input/datasets/eljazouly/ip102-coco-annotations/coco_annotations/train.json'
with open(ann_path, 'r') as f:
    coco_data = json.load(f)

categories = sorted(coco_data['categories'], key=lambda x: x['id'])
class_names = [cat['name'] for cat in categories]
num_classes = len(class_names)
print(f"Tổng số lớp học (classes): {num_classes}")

# 6) Lưu danh sách class vào file class_texts.json
class_texts = [[name] for name in class_names]
with open('data/texts/IP102/class_texts.json', 'w') as f:
    json.dump(class_texts, f)

# 7) Nạp CLIP Model cục bộ từ Kaggle Input
print("-> Đang nạp CLIP model cục bộ từ Kaggle Input...")
clip_local_dir = 'pretrained_models/clip-vit-base-patch32'
clip_configs = glob.glob('/kaggle/input/**/config.json', recursive=True)
kaggle_input_clip = None
for cfg in clip_configs:
    if 'clip-vit-base-patch32' in cfg or 'vit-base-patch32' in cfg:
        kaggle_input_clip = os.path.dirname(cfg)
        break

if kaggle_input_clip:
    print(f"-> Tìm thấy CLIP model tại: {kaggle_input_clip}")
    !cp -rf {kaggle_input_clip}/* pretrained_models/clip-vit-base-patch32/
    print("-> Đã nạp CLIP model thành công!")
else:
    raise FileNotFoundError("Không tìm thấy model clip-vit-base-patch32 trong Kaggle Input! Hãy nhấn 'Add Input' -> 'Models' -> tìm 'clip-vit-base-patch32' và thêm vào.")

print("-> Đang sinh class embeddings từ CLIP model...")
tokenizer = AutoTokenizer.from_pretrained(clip_local_dir)

# Monkeypatch phòng thủ kép (ghi đè trực tiếp trong modeling_utils)
import transformers.modeling_utils
transformers.modeling_utils.check_torch_load_is_safe = lambda: None

# Nạp model với weights_only=False
model = CLIPTextModelWithProjection.from_pretrained(clip_local_dir, use_safetensors=False, weights_only=False)
model.eval()

embeddings = []
with torch.no_grad():
    for name in class_names:
        inputs = tokenizer(name, padding=True, return_tensors="pt")
        outputs = model(**inputs)
        embed = outputs.text_embeds[0].cpu().numpy()
        embed = embed / np.linalg.norm(embed)
        embeddings.append(embed)

np.save('data/IP102/ip102_gt_embeddings.npy', np.array(embeddings))
print("-> Sinh class embeddings thành công!")

# 8) Sinh file task_att_1_embeddings.pth chứa cả att_embedding và att_text
print("-> Đang sinh file dummy attribute embeddings...")
num_att = num_classes * 25
torch.save({
    'att_embedding': torch.zeros(num_att, 512),
    'att_text': [f"att_{i}" for i in range(num_att)]
}, 'data/IP102/task_att_1_embeddings.pth')

# 9) Sinh file phân phối mẫu dummy
thrs = [0.55]
pos_dist = [{att_i: torch.zeros(10000) for att_i in range(num_att)} for _ in thrs]
neg_dist = [{att_i: torch.zeros(10000) for att_i in range(num_att)} for _ in thrs]
torch.save({
    'positive_distributions': pos_dist,
    'negative_distributions': neg_dist
}, 'data/IP102/mowod_distribution_sim1.pth')

# 10) Chuyển đổi COCO annotations sang PASCAL VOC (XML)
print("-> Đang chuyển đổi COCO JSON sang cấu trúc PASCAL VOC...")
voc_root = 'data/IP102/voc/'
anno_dir = os.path.join(voc_root, "Annotations")
imageset_dir = os.path.join(voc_root, "ImageSets", "Main", "mowod")
os.makedirs(anno_dir, exist_ok=True)
os.makedirs(imageset_dir, exist_ok=True)

image_annos = {}
for ann in coco_data['annotations']:
    image_annos.setdefault(ann['image_id'], []).append(ann)

cat_map = {cat['id']: cat['name'] for cat in categories}
anno_ids = {ann['category_id'] for ann in coco_data['annotations']}

if 0 in anno_ids and 0 not in cat_map:
    if max(anno_ids) == len(class_names) - 1:
        cat_map = {i: class_names[i] for i in range(len(class_names))}
    else:
        cat_map[0] = class_names[0] if len(class_names) > 0 else 'unknown'
        for i in range(1, len(class_names) + 1):
            if i not in cat_map:
                cat_map[i] = class_names[i-1] if i-1 < len(class_names) else 'unknown'

image_names = []
for img in coco_data['images']:
    img_id = img['id']
    file_name = img['file_name']
    width = img.get('width', 640)
    height = img.get('height', 640)
    img_stem = os.path.splitext(os.path.basename(file_name))[0]
    image_names.append(img_stem)
    
    xml_content = f"""<annotation>
  <filename>{os.path.basename(file_name)}</filename>
  <size>
    <width>{width}</width>
    <height>{height}</height>
    <depth>3</depth>
  </size>
"""
    annos = image_annos.get(img_id, [])
    for ann in annos:
        bbox = ann['bbox']
        cat_name = cat_map.get(ann['category_id'], 'unknown')
        xmin = max(1, int(round(bbox[0])))
        ymin = max(1, int(round(bbox[1])))
        xmax = max(xmin + 1, int(round(bbox[0] + bbox[2])))
        ymax = max(ymin + 1, int(round(bbox[1] + bbox[3])))
        
        xml_content += f"""  <object>
    <name>{cat_name}</name>
    <pose>Unspecified</pose>
    <truncated>0</truncated>
    <difficult>0</difficult>
    <bndbox>
      <xmin>{xmin}</xmin>
      <ymin>{ymin}</ymin>
      <xmax>{xmax}</xmax>
      <ymax>{ymax}</ymax>
    </bndbox>
  </object>
"""
    xml_content += "</annotation>"
    
    xml_path = os.path.join(anno_dir, f"{img_stem}.xml")
    with open(xml_path, 'w', encoding='utf-8') as f_xml:
        f_xml.write(xml_content)

txt_path = os.path.join(imageset_dir, "all_task_test.txt")
with open(txt_path, 'w') as f_txt:
    f_txt.write("\n".join(image_names))

print(f"Thành công chuyển đổi {len(image_names)} nhãn sang VOC XML tại {voc_root}!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 55.9 MB/s eta 0:00:00
  Attempting uninstall: pillow
    Found existing installation: pillow 12.3.0
    Uninstalling pillow-12.3.0:
      Successfully uninstalled pillow-12.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires matplotlib<=3.10,>=3.5, but you have matplotlib 3.11.0 which is incompatible.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.1 which is incompatible.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.5.1 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
bigframes 2.39.0 requires rich<14,>=12.4.4, but you have rich 15.0.0 which is incompatible.
Pillow ver

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

CLIPTextModelWithProjection LOAD REPORT from: pretrained_models/clip-vit-base-patch32
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
visual_projection.weight                                       | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
vision_model

-> Sinh class embeddings thành công!
-> Đang sinh file dummy attribute embeddings...
-> Đang chuyển đổi COCO JSON sang cấu trúc PASCAL VOC...
Thành công chuyển đổi 45095 nhãn sang VOC XML tại data/IP102/voc/!


# abl studies

# Base+GT (Baseline + Ground Truth)

In [8]:
# !PYTHONPATH=. torchrun --nproc_per_node=2 third_party/mmyolo/tools/train.py configs/abl/ip102_t1.py --launcher pytorch

In [9]:
# %cd /kaggle/working/OW_OVD
# !git pull origin master
# import glob
# import os

# # Tự động quét tìm file best weight của Task 1
# weights = glob.glob('work_dirs/ip102_t1/best_*.pth')

# if weights:
#     best_model_path = weights[0]
#     print(f"Đã tìm thấy mô hình tốt nhất: {best_model_path}")
#     # Chạy lệnh Test với 2 GPU và cấu hình mới
#     !PYTHONPATH=. torchrun --nproc_per_node=2 third_party/mmyolo/tools/test.py configs/abl/ip102_t1.py "{best_model_path}" --launcher pytorch
# else:
#     print("Không tìm thấy file trọng số nào. Quá trình Train có thể đã thất bại!")


# All attr (Sử dụng tất cả thuộc tính)

In [10]:
# !PYTHONPATH=. torchrun --nproc_per_node=2 third_party/mmyolo/tools/train.py configs/abl/ip102_t1_all_attr.py --launcher pytorch

In [11]:
# %cd /kaggle/working/OW_OVD
# import glob

# weights = glob.glob('work_dirs/ip102_t1_all_attr/best_*.pth')

# if weights:
#     best_model_path = weights[0]
#     print(f"Đã tìm thấy mô hình tốt nhất của All Attr: {best_model_path}")
#     !PYTHONPATH=. torchrun --nproc_per_node=2 third_party/mmyolo/tools/test.py configs/abl/ip102_t1_all_attr.py "{best_model_path}" --launcher pytorch
# else:
#     print("Không tìm thấy file trọng số nào của All Attr. Quá trình Train có thể đã thất bại!")


# 3

In [12]:
# %env PYTHONPATH=.
# !torchrun --nproc_per_node=2 third_party/mmyolo/tools/train.py configs/abl/ip102_t1_all_attr_ood.py --launcher pytorch

In [13]:
# %cd /kaggle/working/OW_OVD
# import glob

# # Auto find the best checkpoint generated by configs/abl/ip102_t1_all_attr_ood.py
# weights = glob.glob('work_dirs/ip102_t1_all_attr_ood/best_*.pth')

# if weights:
#     best_model_path = weights[0]
#     print(f'Found best checkpoint for All Attr + OOD Prob: {best_model_path}')
#     # Run test with 2 GPUs
#     !PYTHONPATH=. torchrun --nproc_per_node=2 third_party/mmyolo/tools/test.py configs/abl/ip102_t1_all_attr_ood.py "{best_model_path}" --launcher pytorch
# else:
#     print('No checkpoint weight found. Training may have failed!')

In [14]:
# !python -c "import importlib.util; import re; [open(spec.origin, 'w').write(re.sub(r'mmcv_maximum_version\s*=\s*[\'\"](.*?)[\'\"]', 'mmcv_maximum_version = \'2.3.0\'', open(spec.origin, 'r').read())) for pkg in ['mmdet', 'mmyolo'] for spec in [importlib.util.find_spec(pkg)] if spec and spec.origin]"


# abaltion 5

In [15]:
# %cd /kaggle/working/OW_OVD
# %env PYTHONPATH=.

# !torchrun --nproc_per_node=2 third_party/mmyolo/tools/train.py configs/abl/ip102_t1_sim_restr.py --launcher pytorch

In [16]:
# %cd /kaggle/working/OW_OVD
# import glob

# weights = glob.glob('work_dirs/ip102_t1_sim_restr/best_*.pth')

# if weights:
#     best_model_path = weights[0]
#     print(f'Tìm thấy checkpoint Sim Restr: {best_model_path}')
#     !PYTHONPATH=. torchrun --nproc_per_node=2 third_party/mmyolo/tools/test.py configs/abl/ip102_t1_sim_restr.py "{best_model_path}" --launcher pytorch
# else:
#     print('Ko thấy checkpoint, có lẽ đã fail từ bước train!')

# !PYTHONPATH=. torchrun --nproc_per_node=2 third_party/mmyolo/tools/test.py configs/abl/ip102_t1_sim_restr.py "{best_model_path}" --launcher pytorch


# ablation 6

In [17]:
# !PYTHONPATH=. torchrun --nproc_per_node=2 third_party/mmyolo/tools/train.py configs/abl/ip102_t1_known_uncer.py --launcher pytorch

In [18]:
# %cd /kaggle/working/OW_OVD
# import glob

# # 2. Tự động quét tìm file checkpoint tốt nhất được sinh ra từ configs/abl/ip102_t1_known_uncer.py
# weights = glob.glob('work_dirs/ip102_t1_known_uncer/best_*.pth')

# if weights:
#     best_model_path = weights[0]
#     print(f"Đã tìm thấy mô hình tốt nhất của Known Uncer: {best_model_path}")
    
#     # Chạy lệnh Test với 2 GPU sử dụng module run của môi trường Python hiện tại
#     # Override att_embeddings bằng file sim_restr (kế thừa từ cơ chế 5) để khớp kích thước checkpoint
#     !PYTHONPATH=. python -m torch.distributed.run --nproc_per_node=2 third_party/mmyolo/tools/test.py \
#         configs/abl/ip102_t1_known_uncer.py \
#         "{best_model_path}" \
#         --cfg-options model.bbox_head.att_embeddings=data/IP102/selected_att_embeddings_sim_restr.pth \
#         --launcher pytorch
# else:
#     print("Không tìm thấy file trọng số nào của Known Uncer. Quá trình Train có thể đã thất bại!")


# 4444

In [19]:
%cd /kaggle/working/OW_OVD
%env PYTHONPATH=.

!torchrun --nproc_per_node=2 third_party/mmyolo/tools/train.py configs/abl/ip102_t1_attr_sel.py --launcher pytorch

/kaggle/working/OW_OVD
env: PYTHONPATH=.
W0715 12:23:51.582000 139705726198912 torch/distributed/run.py:779] 
W0715 12:23:51.582000 139705726198912 torch/distributed/run.py:779] *****************************************
W0715 12:23:51.582000 139705726198912 torch/distributed/run.py:779] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0715 12:23:51.582000 139705726198912 torch/distributed/run.py:779] *****************************************
/usr/local/lib/python3.12/dist-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \
/usr/local/lib/python3.12/dist-packages/mmengine/optim/optimizer/zero_optimizer.

In [ ]:
%cd /kaggle/working/OW_OVD
%env PYTHONPATH=.

import os
import glob
from PIL import Image, ImageDraw, ImageFont
from mmdet.apis import init_detector, inference_detector

# ------------------------------------------------------------
# 1) Cấu hình đường dẫn model do bạn tự điền
# ------------------------------------------------------------
CFG_PATH = 'configs/abl/ip102_t1_attr_sel.py'     # <-- đổi theo config bạn dùng
CKPT_PATH = 'work_dirs/ip102_t1_attr_sel/best_*.pth'  # <-- đổi theo checkpoint bạn dùng

# Nếu checkpoint chưa đặt đúng, tìm tự động trong work_dirs
if '*' in CKPT_PATH:
    cand = sorted(glob.glob(CKPT_PATH, recursive=True))
    if cand:
        CKPT_PATH = cand[0]
        print('Using detected checkpoint:', CKPT_PATH)
    else:
        raise FileNotFoundError(f'Không tìm thấy checkpoint theo pattern: {CKPT_PATH}')

if not os.path.exists(CFG_PATH):
    raise FileNotFoundError(f'Không tìm thấy config: {CFG_PATH}')
if not os.path.exists(CKPT_PATH):
    raise FileNotFoundError(f'Không tìm thấy checkpoint: {CKPT_PATH}')

# ------------------------------------------------------------
# 2) Tải model
# ------------------------------------------------------------
model = init_detector(CFG_PATH, CKPT_PATH, device='cuda:0')

# Tên lớp có trong dataset_meta; nếu không có thì mặc định rỗng
class_names = []
if hasattr(model, 'dataset_meta') and isinstance(model.dataset_meta, dict):
    class_names = model.dataset_meta.get('classes', [])

# Nếu model dùng label unknown = 80 trong pipeline OpenWorld, giữ như fallback
UNKNOWN_ID = 80

# ------------------------------------------------------------
# 3) Lấy 10 ảnh đầu từ test folder
# ------------------------------------------------------------
TEST_ROOT = '/kaggle/input/datasets/rtlmhjbn/ip02-dataset/classification/test/0'
OUT_DIR = '/kaggle/working/OW_OVD/paper_visuals'
os.makedirs(OUT_DIR, exist_ok=True)

img_paths = []
for ext in ('*.jpg', '*.jpeg', '*.png', '*.bmp'):
    img_paths.extend(sorted(glob.glob(os.path.join(TEST_ROOT, ext))))
img_paths = img_paths[:10]

if not img_paths:
    raise FileNotFoundError(f'Không tìm thấy ảnh nào trong: {TEST_ROOT}')

print('Found', len(img_paths), 'images in test folder.')
for p in img_paths[:3]:
    print('Sample:', p)

# ------------------------------------------------------------
# 4) Vẽ bounding box lên 10 ảnh
# ------------------------------------------------------------
try:
    font = ImageFont.truetype('arial.ttf', 24)
except Exception:
    font = ImageFont.load_default()

for idx, img_path in enumerate(img_paths, start=1):
    result = inference_detector(model, img_path)
    pred_instances = result.pred_instances

    boxes = pred_instances.bboxes.cpu().numpy()
    scores = pred_instances.scores.cpu().numpy()
    labels = pred_instances.labels.cpu().numpy().astype(int)

    img = Image.open(img_path).convert('RGB')
    draw = ImageDraw.Draw(img)

    for box, score, label_id in zip(boxes, scores, labels):
        x1, y1, x2, y2 = [int(v) for v in box]
        is_unknown = (label_id == UNKNOWN_ID) or (label_id >= len(class_names))

        if is_unknown:
            label_name = 'unknown'
            color = (255, 0, 0)   # đỏ
        else:
            label_name = class_names[label_id] if label_id < len(class_names) else f'class_{label_id}'
            color = (255, 215, 0) # vàng

        draw.rectangle([x1, y1, x2, y2], outline=color, width=3)
        draw.text((max(1, x1), max(1, y1 - 28)), f'{label_name} {score:.2f}', fill=color, font=font)

    out_path = os.path.join(OUT_DIR, f'pred_{idx:02d}.png')
    img.save(out_path)
    print('Saved:', out_path)

print('Done. Visuals are in:', OUT_DIR)


/kaggle/working/OW_OVD
Không tìm thấy file trọng số nào của Attr Sel. Quá trình Train có thể đã thất bại!
